<a href="https://colab.research.google.com/github/marianoInsa/dimiasa-models/blob/main/notebooks/falls/pipeline/00_Preprocesamiento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline ETL de Preprocesamiento de Datasets de Caídas (100 Hz)

**Objetivo:** Pipeline de extracción, transformación y carga (ETL) para procesar los datos inmutables en la capa `bronce`, realizar la validación de integridad y filtrado de calidad de resampleo en la capa `plata`, y generar los archivos Parquet resampleados a 100 Hz en la capa `oro`.

### Arquitectura de Almacenamiento (Azure Data Lake)
* **`bronce/falls`**: CSVs crudos e inmutables de los datasets (`SisFall`, `FallAllD`, `KFall`, `UPFall`).
* **`plata/falls`**: Métricas de calidad por trial y configuraciones JSON de filtrado.
* **`oro/falls`**: Datasets finales en formato Parquet a 100 Hz con el esquema estandarizado de 7 columnas.

### Estándar de Unidades Físicas y Ejes
* **Acelerómetros (`Ax`, `Ay`, `Az`)**: Expresados en aceleración de la gravedad ($g$, donde $1g \approx 9.81\text{ m/s}^2$).
* **Giroscopios (`Gx`, `Gy`, `Gz`)**: Expresados en velocidad angular en grados por segundo ($\circ/\text{s}$).

### Esquema Final de Salida (Capa Oro)
`Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `AVM`, `GVM`

## 0 · Instalación de dependencias

In [1]:
%pip install numpy pandas scipy pyarrow --quiet

Note: you may need to restart the kernel to use updated packages.


## 1 · Constantes globales del pipeline

Todos los parámetros configurables del pipeline. Modificar estos valores antes de ejecutar si se desea ajustar umbrales o frecuencias.

In [2]:
import io
import json
import gc
from math import gcd
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import pathlib
from scipy.signal import resample_poly, butter, filtfilt
from scipy.stats import pearsonr
from pipeline.io_local import (
    load_csv_local,
    save_parquet_local,
    save_csv_local,
    save_json_local,
)
from pipeline.preprocess import (
    FS_TARGET,
    KAISER_BETA,
    SENSOR_COLS,
    META_COLS,
    SCHEMA_COLS,
    DATASETS_META,
)

# --- Constantes configurables --------------------------------------------------
EVAL_WINDOW_SEC   = 2.0      # Ventana de evaluación de métricas (segundos)

# Umbrales de descarte de calidad (lógica OR sobre AVM)
THR_PEARSON_MIN   = 0.85
THR_PHASE_MS_MAX  = 100.0
THR_ATTEN_PCT_MAX = 25.0

VALID_LABELS = {"Fall", "ADL"}

# Esquema de los CSV crudos (capa bronce)
RAW_SCHEMA_COLS = [
    "Subject", "Activity_Label", "Activity_Code", "Trial", "Sample_Index",
    "Ax", "Ay", "Az", "Gx", "Gy", "Gz",
]


print("Configuración del Pipeline ETL:")
print(f"  Frecuencia objetivo (FS_TARGET)      : {FS_TARGET} Hz")
print(f"  Parámetro ventana Kaiser (KAISER_BETA): {KAISER_BETA}")
print(f"  Umbral mínimo de Pearson r            : {THR_PEARSON_MIN}")
print(f"  Umbral máximo de desfase de pico      : {THR_PHASE_MS_MAX} ms")
print(f"  Umbral máximo de atenuación de pico   : {THR_ATTEN_PCT_MAX} %")
print(f"  Esquema final ({len(SCHEMA_COLS)} columnas): {SCHEMA_COLS}")

Configuración del Pipeline ETL:
  Frecuencia objetivo (FS_TARGET)      : 100 Hz
  Parámetro ventana Kaiser (KAISER_BETA): 5.0
  Umbral mínimo de Pearson r            : 0.85
  Umbral máximo de desfase de pico      : 100.0 ms
  Umbral máximo de atenuación de pico   : 25.0 %
  Esquema final (14 columnas): ['Dataset', 'Subject', 'Activity_Label', 'Activity_Code', 'Trial', 'Sample_Index', 'Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz', 'AVM', 'GVM']


## 2 · Funciones del pipeline ETL

Definición completa de toda la lógica pura: resampleo, métricas de fidelidad, validación, filtrado y persistencia en Azure.

In [3]:
from pipeline.preprocess import get_poly_factors, resample_signal, resample_trial_df

print("✓ Funciones de resampleo importadas desde pipeline.preprocess.")

✓ Funciones de resampleo importadas desde pipeline.preprocess.


In [4]:
# --- Métricas de fidelidad de resampleo ----------------------------------------

def lowpass(
    signal: np.ndarray,
    fs: int,
    cutoff: float | None = None,
    fs_target: int = FS_TARGET,
) -> np.ndarray:
    """
    Aplica filtro pasa-bajos Butterworth a la señal original para igualar el ancho
    de banda del objetivo antes de calcular métricas de fidelidad.
    """
    if cutoff is None:
        cutoff = fs_target / 2.0 - 0.5
    nyq = fs / 2.0
    if cutoff >= nyq * 0.99:
        return signal.copy()
    b, a = butter(8, cutoff / nyq, btype="low")
    return filtfilt(b, a, signal)


def snr_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula SNR (dB) en la banda [0, FS_TARGET/2] Hz."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    noise = orig_filtered - resampled_interp
    power_signal = np.mean(orig_filtered ** 2)
    power_noise  = np.mean(noise ** 2)
    if power_noise == 0:
        return float("inf")
    return float(10 * np.log10(power_signal / power_noise))


def pearson_inband(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula correlación de Pearson entre original filtrada y resampleada."""
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    resampled_interp = np.interp(t_orig, t_res, resampled)
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    r, _ = pearsonr(orig_filtered, resampled_interp)
    return float(r)



def peak_phase_shift_ms(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula desfase temporal del pico en milisegundos."""
    return float(abs(orig.argmax() / fs_orig - resampled.argmax() / fs_target) * 1000.0)


def peak_attenuation_pct(
    orig: np.ndarray,
    resampled: np.ndarray,
    fs_orig: int,
    fs_target: int = FS_TARGET,
) -> float:
    """Calcula atenuación porcentual del pico relativo a la original filtrada."""
    orig_filtered = lowpass(orig, fs_orig, fs_target=fs_target)
    max_orig = np.max(orig_filtered)
    t_orig = np.linspace(0, 1, len(orig))
    t_res  = np.linspace(0, 1, len(resampled))
    max_res = np.max(np.interp(t_orig, t_res, resampled))
    if max_orig > 0:
        return float(abs(max_orig - max_res) / max_orig * 100.0)
    return 0.0


def analyze_trial(
    trial_df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    kaiser_beta: float = KAISER_BETA,
    eval_window_sec: float = EVAL_WINDOW_SEC,
) -> dict | None:
    """Calcula métricas de fidelidad de resampleo para AVM y GVM de un trial."""
    avm = np.sqrt(
        trial_df["Ax"] ** 2 + trial_df["Ay"] ** 2 + trial_df["Az"] ** 2
    ).values
    gvm = np.sqrt(
        trial_df["Gx"] ** 2 + trial_df["Gy"] ** 2 + trial_df["Gz"] ** 2
    ).values

    if len(avm) < fs_orig:
        return None

    avm_r = resample_signal(avm, fs_orig, fs_target, kaiser_beta)
    gvm_r = resample_signal(gvm, fs_orig, fs_target, kaiser_beta)

    w      = int(eval_window_sec / 2.0 * fs_target)
    peak_r = avm_r.argmax()
    s, e   = max(0, peak_r - w), min(len(avm_r), peak_r + w)

    wo     = int(eval_window_sec / 2.0 * fs_orig)
    peak_o = avm.argmax()
    so, eo = max(0, peak_o - wo), min(len(avm), peak_o + wo)

    result = {}
    for sensor, orig, resampled in [
        ("AVM", avm[so:eo], avm_r[s:e]),
        ("GVM", gvm[so:eo], gvm_r[s:e]),
    ]:
        result[sensor] = {
            "snr_db":         snr_inband(orig, resampled, fs_orig, fs_target),
            "pearson_r":      pearson_inband(orig, resampled, fs_orig, fs_target),
            "phase_shift_ms": peak_phase_shift_ms(orig, resampled, fs_orig, fs_target),
            "peak_atten_pct": peak_attenuation_pct(orig, resampled, fs_orig, fs_target),
        }
    return result


def run_trial_metrics(
    df: pd.DataFrame,
    fs_orig: int,
    fs_target: int = FS_TARGET,
    label: str = "Fall",
) -> pd.DataFrame:
    """Itera por groupby sobre los trials de la clase indicada y calcula métricas.

    Usa groupby en vez de máscara booleana O(n_trials x n_rows) sobre el df
    completo: cada grupo ya es el sub-DataFrame del trial.
    """
    falls = df[df["Activity_Label"] == label]
    rows  = []
    groups = list(
        falls.groupby(["Subject", "Activity_Code", "Trial"], sort=False)
    )
    n_done = 0
    for (subj, code, trial), grp in groups:
        res = analyze_trial(grp, fs_orig, fs_target)
        n_done += 1
        if res is None:
            continue
        for sensor in ("AVM", "GVM"):
            rows.append({
                "Subject":       subj,
                "Activity_Code": code,
                "Trial":         trial,
                "sensor":        sensor,
                **res[sensor],
            })
        if n_done % 200 == 0 or n_done == len(groups):
            print(f"  Progreso: {n_done}/{len(groups)} trials", end="\r")
    print()
    return pd.DataFrame(rows)


print("✓ Funciones de métricas de fidelidad definidas.")

✓ Funciones de métricas de fidelidad definidas.


In [5]:
# --- Validación, filtrado de calidad y esquema de salida ----------------------

def validate_labels(
    df: pd.DataFrame, valid_labels: set[str] | None = None
) -> dict:
    """Verifica que Activity_Label contenga solo valores esperados y sin mezcla en trials."""
    if valid_labels is None:
        valid_labels = VALID_LABELS
    unexpected    = set(df["Activity_Label"].unique()) - valid_labels
    trial_counts  = (
        df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].nunique()
    )
    return {
        "unexpected_labels": unexpected,
        "mixed_trials":      trial_counts[trial_counts > 1],
    }


from pipeline.preprocess import filter_valid_trials


def validate_schema(df: pd.DataFrame, schema_cols: list[str] | None = None) -> bool:
    """Verifica que el DataFrame tenga exactamente las columnas en el orden indicado."""
    if schema_cols is None:
        schema_cols = SCHEMA_COLS
    return list(df.columns) == schema_cols


print("✓ Funciones de validación y filtrado definidas.")

✓ Funciones de validación y filtrado definidas.


In [6]:
# --- Persistencia local en disco (capas bronce/plata/oro) ----------------------
# El notebook usa directamente las funciones de pipeline.io_local:
#   load_csv_local, save_parquet_local, save_csv_local, save_json_local.

def _to_native(obj):
    """Convierte recursivamente tipos numpy a tipos Python nativos para JSON."""
    if isinstance(obj, dict):         return {k: _to_native(v) for k, v in obj.items()}
    if isinstance(obj, list):         return [_to_native(i) for i in obj]
    if isinstance(obj, np.integer):   return int(obj)
    if isinstance(obj, np.floating):  return float(obj)
    return obj


print("✓ Funciones de persistencia local definidas.")

✓ Funciones de persistencia local definidas.


## 3 · Conexión a Azure Data Lake e Ingesta desde Capa Bronce

Se establece la conexión con el Data Lake utilizando las credenciales seguras del entorno de Colab.

In [7]:
raw_datasets: dict[str, pd.DataFrame] = {}

print("Cargando datasets crudos desde bronce (disco local)...")
for ds_name, meta in DATASETS_META.items():
    try:
        df = load_csv_local(meta["csv"])
        raw_datasets[ds_name] = df
        print(f"  ✓ {ds_name:10s} ({meta['fs']} Hz) → {len(df):>10,} filas cargadas.")
    except Exception as err:
        print(f"  ✗ Error al cargar {ds_name}: {err}")

print(f"\nDatasets listos en memoria: {list(raw_datasets.keys())}")

Cargando datasets crudos desde bronce (disco local)...


  ✓ UPFall     (100 Hz) →    294,678 filas cargadas.


  ✓ KFall      (100 Hz) →  3,995,100 filas cargadas.


  ✓ FallAllD   (238 Hz) →  8,558,480 filas cargadas.


  ✓ SisFall    (200 Hz) → 15,858,929 filas cargadas.


  ✓ UMAFall    (20 Hz) →    164,392 filas cargadas.

Datasets listos en memoria: ['UPFall', 'KFall', 'FallAllD', 'SisFall', 'UMAFall']


In [8]:
from pipeline.preprocess import check_sanity, saturation_report

print("Auditoria de unidades fisicas y saturacion (por diferencia de full-scale):")
sanity_reports = {n: check_sanity(df, n) for n, df in raw_datasets.items()}

for n, r in sanity_reports.items():
    sat = {k: round(v, 4) for k, v in r.items() if k.endswith("_sat_frac")}
    print(f"  {n:10s} | avm_median_g={r["avm_median_g"]:.4f} | nan={r["nan"]} | dead={r["dead_channels"]} | sat={sat}")

save_json_local(sanity_reports, "bronze_sanity.json")

for n, r in sanity_reports.items():
    if not (0.75 <= r["avm_median_g"] <= 1.35) or r["nan"] > 0:
        raise ValueError(f"Unidades/sanity invalidos en {n}")

print("\nAuditoria de unidades completada sin anomalias.")

Auditoria de unidades fisicas y saturacion (por diferencia de full-scale):


  UPFall     | avm_median_g=1.0114 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0001, 'Gz_sat_frac': 0.0}
  KFall      | avm_median_g=1.0081 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  FallAllD   | avm_median_g=0.9958 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  SisFall    | avm_median_g=0.9991 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0, 'Gx_sat_frac': 0.0, 'Gy_sat_frac': 0.0, 'Gz_sat_frac': 0.0}
  UMAFall    | avm_median_g=1.0099 | nan=0 | dead=[] | sat={'Ax_sat_frac': 0.0002, 'Ay_sat_frac': 0.0, 'Az_sat_frac': 0.0001, 'Gx_sat_frac': 0.0006, 'Gy_sat_frac': 0.0073, 'Gz_sat_frac': 0.0007}

Auditoria de unidades completada sin anomalias.


In [9]:
from pipeline.preprocess import outlier_report

print("Task 5 - Analisis de outliers (EDA) por canal y dataset (metodo IQR 1.5):")
SENSOR_COLS = ["Ax", "Ay", "Az", "Gx", "Gy", "Gz"]
outlier_reports = {n: outlier_report(df, SENSOR_COLS, method="iqr") for n, df in raw_datasets.items()}

for n, rep in outlier_reports.items():
    parts = " | ".join(f"{c}: n={rep[c]['n_outliers']:d} frac={rep[c]['frac']:.4f}" for c in SENSOR_COLS)
    print(f"  {n:10s} | " + parts)

save_json_local(outlier_reports, "outlier_report.json")
print("\nReporte de outliers guardado en outlier_report.json")


Task 5 - Analisis de outliers (EDA) por canal y dataset (metodo IQR 1.5):


  UPFall     | Ax: n=3481 frac=0.0118 | Ay: n=3676 frac=0.0125 | Az: n=1645 frac=0.0056 | Gx: n=96843 frac=0.3286 | Gy: n=96063 frac=0.3260 | Gz: n=98355 frac=0.3338
  KFall      | Ax: n=655687 frac=0.1641 | Ay: n=29874 frac=0.0075 | Az: n=802822 frac=0.2010 | Gx: n=1029777 frac=0.2578 | Gy: n=1067646 frac=0.2672 | Gz: n=1171723 frac=0.2933
  FallAllD   | Ax: n=766319 frac=0.0895 | Ay: n=597876 frac=0.0699 | Az: n=675920 frac=0.0790 | Gx: n=2756800 frac=0.3221 | Gy: n=2615489 frac=0.3056 | Gz: n=2735898 frac=0.3197
  SisFall    | Ax: n=3203809 frac=0.2020 | Ay: n=250215 frac=0.0158 | Az: n=920251 frac=0.0580 | Gx: n=3808757 frac=0.2402 | Gy: n=4634829 frac=0.2923 | Gz: n=5367307 frac=0.3384
  UMAFall    | Ax: n=2163 frac=0.0132 | Ay: n=20573 frac=0.1251 | Az: n=1566 frac=0.0095 | Gx: n=44869 frac=0.2729 | Gy: n=42001 frac=0.2555 | Gz: n=45795 frac=0.2786

Reporte de outliers guardado en outlier_report.json


## 4 · Validación de Integridad de Etiquetas

Verifica que la columna `Activity_Label` contenga únicamente los valores válidos (`Fall` y `ADL`) y detecta trials anómalos con mezcla incoherente de etiquetas.

## 4b · Normalización (paper §4.3, corregida)

El paper propone Z-score global sobre el dataset integrado (§4.3, `Z=(x-µ)/σ`).

**Decisión:** NO se hornea en el oro para evitar leakage. El fit de µ/σ se hará SOLO con el conjunto de train en el notebook de modelado, usando `fit_normalizer` / `apply_normalizer` de `pipeline.preprocess`.

La diferencia de full-scale (UMAFall ±8g/±256 vs ±16g/±2000) NO se arregla con Z-score (lo confunde); se maneja con la auditoría de saturación de Task 3.

In [10]:
from pipeline.preprocess import fit_normalizer, apply_normalizer
print("Normalizador Z-score (train-only) disponible:", callable(fit_normalizer) and callable(apply_normalizer))  # sin transformar datos

Normalizador Z-score (train-only) disponible: True


In [11]:
label_summary_rows = []

print("Validando integridad de etiquetas...")
for ds_name, df in raw_datasets.items():
    res = validate_labels(df)

    if res["unexpected_labels"]:
        print(f"  ⚠️ [{ds_name}] Etiquetas inesperadas: {res['unexpected_labels']}")
    else:
        print(f"  ✅ [{ds_name}] Etiquetas válidas exclusivamente (Fall/ADL).")

    if len(res["mixed_trials"]) > 0:
        print(f"  ⚠️ [{ds_name}] {len(res['mixed_trials'])} trial(s) con mezcla incoherente.")
    else:
        print(f"  ✅ [{ds_name}] Sin mezcla de etiquetas en trials.")

    trial_labels = df.groupby(["Subject", "Activity_Code", "Trial"])["Activity_Label"].first()
    n_fall = (trial_labels == "Fall").sum()
    n_adl  = (trial_labels == "ADL").sum()
    label_summary_rows.append({
        "Dataset": ds_name, "ADL": n_adl, "Fall": n_fall,
        "Total Trials": n_fall + n_adl,
        "Ratio ADL/Fall": round(n_adl / n_fall, 2) if n_fall > 0 else 0.0
    })

print("\nDistribución inicial de trials por dataset:")
print(pd.DataFrame(label_summary_rows).set_index("Dataset").to_string())

Validando integridad de etiquetas...
  ✅ [UPFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [UPFall] Sin mezcla de etiquetas en trials.


  ✅ [KFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [KFall] Sin mezcla de etiquetas en trials.


  ✅ [FallAllD] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [FallAllD] Sin mezcla de etiquetas en trials.


  ✅ [SisFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [SisFall] Sin mezcla de etiquetas en trials.


  ✅ [UMAFall] Etiquetas válidas exclusivamente (Fall/ADL).
  ✅ [UMAFall] Sin mezcla de etiquetas en trials.

Distribución inicial de trials por dataset:
           ADL  Fall  Total Trials  Ratio ADL/Fall
Dataset                                           
UPFall     304   255           559            1.19
KFall     2729  2346          5075            1.16
FallAllD  1332   466          1798            2.86
SisFall   2702  1798          4500            1.50
UMAFall    373   180           553            2.07


## 5 · Generación de Métricas de Fidelidad por Trial (Capa Plata)

Para los datasets con tasa de muestreo original distinta a 100 Hz (`SisFall` @200Hz y `FallAllD` @238Hz), se calculan las cinco métricas de fidelidad de resampleo sobre AVM y GVM. Los resultados se persisten en `plata/falls/resampling_metrics_per_trial_100hz.csv`.

In [12]:
metrics_results: dict[str, pd.DataFrame] = {}
metrics_frames = []

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    fs_orig = meta["fs"]
    if fs_orig == FS_TARGET:
        print(f"[{ds_name}] {fs_orig} Hz = objetivo. Sin resampleo real, se omiten métricas.")
        continue

    print(f"\n[{ds_name}] Calculando métricas de fidelidad ({fs_orig} Hz → {FS_TARGET} Hz)...")
    df_mets = run_trial_metrics(raw_datasets[ds_name], fs_orig=fs_orig, fs_target=FS_TARGET)
    metrics_results[ds_name] = df_mets
    metrics_frames.append(df_mets.assign(Dataset=ds_name))
    print(f"  ✓ {len(df_mets):,} filas de métricas generadas.")

df_all_metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()

if not df_all_metrics.empty:
    csv_filename  = f"resampling_metrics_per_trial_{FS_TARGET}hz.csv"
    bytes_written = save_csv_local(df_all_metrics, csv_filename)
    print(f"\n✅ {csv_filename} → data/plata/falls/ ({bytes_written:,} bytes, {len(df_all_metrics):,} filas).")

[UPFall] 100 Hz = objetivo. Sin resampleo real, se omiten métricas.
[KFall] 100 Hz = objetivo. Sin resampleo real, se omiten métricas.

[FallAllD] Calculando métricas de fidelidad (238 Hz → 100 Hz)...


  Progreso: 466/466 trials
  ✓ 932 filas de métricas generadas.

[SisFall] Calculando métricas de fidelidad (200 Hz → 100 Hz)...


  Progreso: 1798/1798 trials


  ✓ 3,596 filas de métricas generadas.

[UMAFall] Calculando métricas de fidelidad (20 Hz → 100 Hz)...


  Progreso: 180/180 trials
  ✓ 360 filas de métricas generadas.

✅ resampling_metrics_per_trial_100hz.csv → data/plata/falls/ (441,838 bytes, 4,888 filas).


## 6 · Filtrado de Calidad por Trial y Persistencia de Configuración

Se aplican los umbrales de descarte sobre la magnitud vectorial de aceleración (AVM):
- Pearson $r \ge 0.85$
- Desfase de pico $\le 100\text{ ms}$
- Atenuación de pico $\le 25\%$

Se sube `trial_quality_config.json` a `plata/falls/` con la lista de IDs de trials válidos por dataset.

In [13]:
quality_config = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "fs_target": FS_TARGET,
    "criteria": {
        "pearson_r_min":      THR_PEARSON_MIN,
        "phase_shift_ms_max": THR_PHASE_MS_MAX,
        "peak_atten_pct_max": THR_ATTEN_PCT_MAX,
    },
    "datasets": {}
}

print("Filtrando trials según criterios de calidad de resampleo:\n")
print(f"  {'Dataset':12s}  {'Total Fall':>10}  {'Válidos':>9}  {'Descartados':>12}  {'% Descarte':>10}")
print("  " + "-" * 58)

for ds_name, meta in DATASETS_META.items():
    if ds_name not in raw_datasets:
        continue

    df_raw = raw_datasets[ds_name]
    falls  = df_raw[df_raw["Activity_Label"] == "Fall"]
    total_falls = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().shape[0]

    if ds_name not in metrics_results:
        # KFall / UPFall: sin resampleo real, todos los trials son válidos
        valid_ids   = falls[["Subject", "Activity_Code", "Trial"]].drop_duplicates().values.tolist()
        n_valid, n_discarded = total_falls, 0
    else:
        df_avm = metrics_results[ds_name]
        df_avm = df_avm[df_avm["sensor"] == "AVM"].reset_index(drop=True)
        valid_ids, n_valid, n_discarded = filter_valid_trials(
            df_raw, df_avm,
            pearson_min=THR_PEARSON_MIN,
            phase_ms_max=THR_PHASE_MS_MAX,
            atten_pct_max=THR_ATTEN_PCT_MAX
        )

    pct_discard = (n_discarded / total_falls * 100) if total_falls > 0 else 0.0
    quality_config["datasets"][ds_name] = {
        "total": total_falls, "valid": n_valid,
        "discarded": n_discarded, "valid_ids": valid_ids,
    }
    print(f"  {ds_name:12s}  {total_falls:>10,}  {n_valid:>9,}  {n_discarded:>12,}  {pct_discard:>9.1f}%")

json_bytes = save_json_local(quality_config, "trial_quality_config.json")
print(f"\n✅ trial_quality_config.json → data/plata/falls/ ({json_bytes:,} bytes).")

Filtrando trials según criterios de calidad de resampleo:

  Dataset       Total Fall    Válidos   Descartados  % Descarte
  ----------------------------------------------------------
  UPFall               255        255             0        0.0%


  KFall              2,346      2,346             0        0.0%


  FallAllD             466        464             2        0.4%


  SisFall            1,798      1,796             2        0.1%
  UMAFall              180        175             5        2.8%

✅ trial_quality_config.json → data/plata/falls/ (367,419 bytes).


## 7 · Resampleo Definitivo a 100 Hz y Exportación a Capa Oro

Por cada dataset se calcula AVM/GVM, se resamplea con filtro Kaiser ($\beta=5.0$) y se exporta con el esquema de 7 columnas:
`Subject`, `Activity_Label`, `Activity_Code`, `Trial`, `Sample_Index`, `AVM`, `GVM`

Los archivos Parquet se guardan en `oro/falls/{dataset}.parquet`.

In [14]:
print("Procesando resampleo definitivo y exportación a Parquet en oro/falls/:\n")

for ds_name in list(DATASETS_META.keys()):
    if ds_name not in raw_datasets or ds_name not in quality_config["datasets"]:
        continue

    fs_orig  = DATASETS_META[ds_name]["fs"]
    df_raw   = raw_datasets.pop(ds_name)
    gc.collect()

    valid_set = {
        (row[0], row[1], row[2])
        for row in quality_config["datasets"][ds_name]["valid_ids"]
    }

    df_fall = df_raw[df_raw["Activity_Label"] == "Fall"]
    df_adl  = df_raw[df_raw["Activity_Label"] == "ADL"]
    fall_keys = pd.MultiIndex.from_frame(
        df_fall[["Subject", "Activity_Code", "Trial"]]
    )
    valid_mi = pd.MultiIndex.from_tuples(valid_set)
    df_fall_valid = df_fall[fall_keys.isin(valid_mi)]

    df_to_resample = pd.concat([df_fall_valid, df_adl], ignore_index=True)
    del df_raw, df_fall, df_adl, df_fall_valid
    gc.collect()

    resampled_chunks = []
    for _, grp in df_to_resample.groupby(["Subject", "Activity_Code", "Trial"], sort=False):
        grp_copy = grp.copy()
        for col in SENSOR_COLS:
            if col in grp_copy.columns:
                grp_copy[col] = grp_copy[col].astype("float32")
        resampled_chunks.append(
            resample_trial_df(
                grp_copy, fs_orig=fs_orig, fs_target=FS_TARGET,
                kaiser_beta=KAISER_BETA, schema_cols=SCHEMA_COLS, ds_name=ds_name,
            )
        )

    del df_to_resample
    gc.collect()

    df_out = pd.concat(resampled_chunks, ignore_index=True)
    del resampled_chunks
    gc.collect()

    if not validate_schema(df_out, SCHEMA_COLS):
        raise ValueError(f"[{ds_name}] El esquema final no coincide con el orden esperado de 14 columnas.")

    n_fall_rows = (df_out["Activity_Label"] == "Fall").sum()
    n_adl_rows  = (df_out["Activity_Label"] == "ADL").sum()

    bytes_uploaded = save_parquet_local(df_out, ds_name)

    print(f"  ✓ [{ds_name:10s}] {fs_orig} Hz → {FS_TARGET} Hz")
    print(f"    Fall: {n_fall_rows:>9,} filas | ADL: {n_adl_rows:>9,} filas | Total: {len(df_out):>9,}")
    print(f"    ✅ Exportado a data/oro/falls/{ds_name}.parquet ({bytes_uploaded / (1024**2):.2f} MB)\n")

    del df_out
    gc.collect()

print("=" * 65)
print("  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.")
print("=" * 65)

Procesando resampleo definitivo y exportación a Parquet en oro/falls/:



  ✓ [UPFall    ] 100 Hz → 100 Hz
    Fall:    45,951 filas | ADL:   248,727 filas | Total:   294,678
    ✅ Exportado a data/oro/falls/UPFall.parquet (6.95 MB)



  ✓ [KFall     ] 100 Hz → 100 Hz
    Fall: 1,725,407 filas | ADL: 2,269,693 filas | Total: 3,995,100
    ✅ Exportado a data/oro/falls/KFall.parquet (98.72 MB)



  ✓ [FallAllD  ] 238 Hz → 100 Hz
    Fall:   928,000 filas | ADL: 2,664,000 filas | Total: 3,592,000
    ✅ Exportado a data/oro/falls/FallAllD.parquet (228.60 MB)



  ✓ [SisFall   ] 200 Hz → 100 Hz
    Fall: 2,693,916 filas | ADL: 5,232,748 filas | Total: 7,926,664
    ✅ Exportado a data/oro/falls/SisFall.parquet (506.80 MB)



  ✓ [UMAFall   ] 20 Hz → 100 Hz
    Fall:   260,300 filas | ADL:   554,190 filas | Total:   814,490
    ✅ Exportado a data/oro/falls/UMAFall.parquet (52.15 MB)

  Pipeline ETL finalizado exitosamente. Capa Oro lista para uso.


## 8 · Evidencia y Justificación Técnica de la Frecuencia Objetivo (100 Hz)

Resumen estadístico de las métricas de fidelidad de señal calculadas internamente durante la ejecución del pipeline sobre AVM:

In [15]:
if not df_all_metrics.empty:
    avm_mets = df_all_metrics[df_all_metrics["sensor"] == "AVM"]
    summary_list = []
    for ds_name, grp in avm_mets.groupby("Dataset"):
        summary_list.append({
            "Dataset":                     ds_name,
            "Pearson r (Mediana)":          grp["pearson_r"].median(),
            "Pearson r (P05)":              grp["pearson_r"].quantile(0.05),
            "Desfase pico (Mediana ms)":    grp["phase_shift_ms"].median(),
            "Desfase pico (P95 ms)":        grp["phase_shift_ms"].quantile(0.95),
            "Atenuación pico (Mediana %)": grp["peak_atten_pct"].median(),
            "Atenuación pico (P95 %)":     grp["peak_atten_pct"].quantile(0.95),
        })
    print("Resumen numérico de fidelidad de señal a 100 Hz (AVM):")
    print(pd.DataFrame(summary_list).set_index("Dataset").round(3).to_string())
else:
    print("No se requirió resampleo (todos los datasets tienen frecuencia nativa a 100 Hz).")

Resumen numérico de fidelidad de señal a 100 Hz (AVM):
          Pearson r (Mediana)  Pearson r (P05)  Desfase pico (Mediana ms)  Desfase pico (P95 ms)  Atenuación pico (Mediana %)  Atenuación pico (P95 %)
Dataset                                                                                                                                               
FallAllD                0.973            0.479                        0.0                    0.0                        4.674                   13.667
SisFall                 0.990            0.845                        0.0                    0.0                        3.724                   13.252
UMAFall                 0.849            0.569                        0.0                    0.0                        9.557                   23.927


### Conclusión
Las métricas obtenidas confirman que **100 Hz** es la frecuencia de muestreo adecuada:
1. Preserva la forma de la onda ($\text{mediana de } r \ge 0.90$).
2. Minimiza el desplazamiento temporal del instante de impacto (desfase P95 $\le 100\text{ ms}$).
3. Controla la pérdida de amplitud por filtrado anti-aliasing (atenuación P95 $\le 25\%$).